In [5]:
from datasets import load_dataset
from collections import Counter
import numpy as np
import pandas as pd
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

In [6]:
dataset=load_dataset("imdb")
print(dataset)
print(dataset['train'][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and

In [7]:
print("Train size:", len(dataset['train']))
print("Test size:", len(dataset['test']))

print("Train label counts:", Counter(dataset['train']["label"]))
print("Test label counts:", Counter(dataset['test']["label"]))

Train size: 25000
Test size: 25000
Train label counts: Counter({0: 12500, 1: 12500})
Test label counts: Counter({0: 12500, 1: 12500})


In [8]:
train_texts=dataset['train']['text']

char_lengths=[len(text) for text in train_texts]

print("Min char length:", min(char_lengths))
print("Max char length:", max(char_lengths))
print("Mean char length:", sum(char_lengths)/len(char_lengths))

Min char length: 52
Max char length: 13704
Mean char length: 1325.06964


In [9]:
print("50% quantile:", np.percentile(char_lengths, 50))
print("90% quantile:", np.percentile(char_lengths, 90))
print("95% quantile:", np.percentile(char_lengths, 95))
print("99% quantile:", np.percentile(char_lengths, 99))

50% quantile: 979.0
90% quantile: 2617.0
95% quantile: 3432.0499999999993
99% quantile: 5213.009999999998


In [10]:
print(dataset['train'].features['label'])

ClassLabel(names=['neg', 'pos'])


In [11]:
train_data=dataset['train']

pos_examples=[example for example in train_data if example['label']==1][:2]
neg_examples=[example for example in train_data if example['label']==0][:2]

print("POSITIVE_EXAMPLES:")
for row in pos_examples:
    print(row['text'][:300])
    print("-"*50)

print("NEGATIVE_EXAMPLES:")
for row in neg_examples:
    print(row['text'])

POSITIVE_EXAMPLES:
Zentropa has much in common with The Third Man, another noir-like film set among the rubble of postwar Europe. Like TTM, there is much inventive camera work. There is an innocent American who gets emotionally involved with a woman he doesn't really understand, and whose naivety is all the more strik
--------------------------------------------------
Zentropa is the most original movie I've seen in years. If you like unique thrillers that are influenced by film noir, then this is just the right cure for all of those Hollywood summer blockbusters clogging the theaters these days. Von Trier's follow-ups like Breaking the Waves have gotten more acc
--------------------------------------------------
NEGATIVE_EXAMPLES:
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of f

In [12]:
train_texts=dataset['train']['text']

br_count=sum("<br" in text.lower() for text in train_texts)
print("Br count:", br_count)
print("Br ratio:", br_count/len(train_texts))

Br count: 14665
Br ratio: 0.5866


In [13]:
short_texts=sum(len(text)<200 for text in train_texts)
print("Short texts count:", short_texts)
print("Short texts ratio:", short_texts/len(train_texts))

Short texts count: 103
Short texts ratio: 0.00412


In [14]:
empty_counts= sum(len(text.strip())==0 for text in train_texts)
print("Empty texts count:", empty_counts)
print("Empty texts ratio:", empty_counts/len(train_texts))

Empty texts count: 0
Empty texts ratio: 0.0


In [15]:
unique_texts=len(set(train_texts))
total_texts=len(train_texts)
duplicate_count=total_texts-unique_texts

print("Total texts count:", total_texts)
print("Unique texts count:", unique_texts)
print("Duplicate texts count:", duplicate_count)
print("Duplicate texts ratio:", duplicate_count/total_texts)

Total texts count: 25000
Unique texts count: 24904
Duplicate texts count: 96
Duplicate texts ratio: 0.00384


In [16]:
train_df=dataset['train'].to_pandas()
test_df=dataset['test'].to_pandas()

print(train_df.head())
print(train_df.shape)

                                                text  label
0  I rented I AM CURIOUS-YELLOW from my video sto...      0
1  "I Am Curious: Yellow" is a risible and preten...      0
2  If only to avoid making this type of film in t...      0
3  This film was probably inspired by Godard's Ma...      0
4  Oh, brother...after hearing about this ridicul...      0
(25000, 2)


In [17]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    25000 non-null  object
 1   label   25000 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 390.8+ KB


In [18]:
train_df["char_length"] = train_df["text"].str.len()
print(train_df["char_length"].describe())

count    25000.00000
mean      1325.06964
std       1003.13367
min         52.00000
25%        702.00000
50%        979.00000
75%       1614.00000
max      13704.00000
Name: char_length, dtype: float64


In [19]:
train_df.groupby("label")["char_length"].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,12500.0,1302.97904,957.067769,52.0,709.0,976.5,1568.0,8969.0
1,12500.0,1347.16024,1046.747365,70.0,695.0,982.0,1651.0,13704.0


In [20]:
def simple_clean(text: str) -> str:
    text=text.lower()
    text = re.sub(r"<br\s*/?>", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text
train_df['clean_text'] = train_df['text'].apply(simple_clean)
print(train_df[["text", "clean_text"]].head())

                                                text  \
0  I rented I AM CURIOUS-YELLOW from my video sto...   
1  "I Am Curious: Yellow" is a risible and preten...   
2  If only to avoid making this type of film in t...   
3  This film was probably inspired by Godard's Ma...   
4  Oh, brother...after hearing about this ridicul...   

                                          clean_text  
0  i rented i am curious yellow from my video sto...  
1  i am curious yellow is a risible and pretentio...  
2  if only to avoid making this type of film in t...  
3  this film was probably inspired by godard s ma...  
4  oh brother after hearing about this ridiculous...  


In [21]:
all_words=" ".join(train_df["clean_text"])
word_counts=Counter(all_words.split())
print("Top 20 words:", word_counts.most_common(20))

Top 20 words: [('the', 336758), ('and', 164143), ('a', 163174), ('of', 145867), ('to', 135724), ('is', 107337), ('it', 96472), ('in', 93981), ('i', 87702), ('this', 76007), ('that', 73287), ('s', 66099), ('was', 48209), ('as', 46940), ('for', 44345), ('with', 44130), ('movie', 44047), ('but', 42624), ('film', 40162), ('t', 34407)]


In [22]:
neg_words=" ".join(train_df[train_df["label"]==0]["clean_text"]).split()
pos_words=" ".join(train_df[train_df["label"]==1]["clean_text"]).split()

neg_counts=Counter(neg_words)
pos_counts=Counter(pos_words)

print("Top 20 negative words:", neg_counts.most_common(20))
print()
print("Top 20 positive words:", pos_counts.most_common(20))

Top 20 negative words: [('the', 163410), ('a', 79408), ('and', 74396), ('of', 69011), ('to', 68975), ('is', 50088), ('it', 48396), ('i', 46921), ('in', 43757), ('this', 40922), ('that', 37640), ('s', 32023), ('was', 26291), ('movie', 24969), ('for', 21928), ('but', 21788), ('with', 20880), ('as', 20629), ('t', 20594), ('film', 19221)]

Top 20 positive words: [('the', 173348), ('and', 89747), ('a', 83766), ('of', 76856), ('to', 66749), ('is', 57249), ('in', 50224), ('it', 48076), ('i', 40781), ('that', 35647), ('this', 35085), ('s', 34076), ('as', 26311), ('with', 23250), ('for', 22417), ('was', 21918), ('film', 20941), ('but', 20836), ('movie', 19078), ('his', 17228)]


In [23]:
stop_words=set(ENGLISH_STOP_WORDS)

neg_filtered=[word for word in neg_words if word not in stop_words and len(word)>1]
pos_filtered=[word for word in pos_words if word not in stop_words and len(word)>1]

neg_filtered_counts=Counter(neg_filtered)
pos_filtered_counts=Counter(pos_filtered)

print("Top 20 negative words (no stop words):", neg_filtered_counts.most_common(20))
print()
print("Top 20 positive words (no stop words):", pos_filtered_counts.most_common(20))

Top 20 negative words (no stop words): [('movie', 24969), ('film', 19221), ('like', 11241), ('just', 10620), ('good', 7423), ('bad', 7401), ('really', 6263), ('time', 6211), ('don', 5363), ('story', 5210), ('people', 4807), ('make', 4722), ('plot', 4155), ('movies', 4081), ('acting', 4056), ('way', 3989), ('think', 3643), ('characters', 3600), ('watch', 3550), ('character', 3508)]

Top 20 positive words (no stop words): [('film', 20941), ('movie', 19078), ('like', 9040), ('good', 7724), ('just', 7154), ('story', 6780), ('time', 6516), ('great', 6419), ('really', 5476), ('people', 4480), ('best', 4320), ('love', 4302), ('life', 4203), ('way', 4037), ('films', 3813), ('think', 3655), ('movies', 3587), ('characters', 3559), ('character', 3516), ('don', 3485)]


In [24]:
common_words= set(neg_filtered_counts.keys()) & set(pos_filtered_counts.keys())

word_diff=[]
for word in common_words:
    neg_counts=neg_filtered_counts[word]
    pos_counts=pos_filtered_counts[word]
    word_diff.append((word, pos_counts-neg_counts))

word_diff_sorted = sorted(word_diff, key=lambda x: x[1])

print("More negative-like words:", word_diff_sorted[:20])
print()
print("More positive-like words:", word_diff_sorted[-20:])

More negative-like words: [('movie', -5891), ('bad', -5494), ('just', -3466), ('worst', -2228), ('like', -2201), ('don', -1878), ('plot', -1721), ('acting', -1618), ('make', -1419), ('awful', -1389), ('minutes', -1299), ('waste', -1260), ('thing', -1203), ('stupid', -1155), ('boring', -1146), ('terrible', -1144), ('script', -1121), ('poor', -1065), ('worse', -1029), ('didn', -981)]

More positive-like words: [('role', 745), ('loved', 746), ('amazing', 797), ('family', 862), ('perfect', 886), ('series', 889), ('man', 948), ('performance', 962), ('beautiful', 986), ('years', 998), ('world', 1025), ('young', 1026), ('wonderful', 1086), ('excellent', 1295), ('story', 1570), ('film', 1720), ('life', 1774), ('love', 2150), ('best', 2224), ('great', 3777)]
